In [1]:
#https://bab2min.github.io/tomotopy/v0.12.2/en/
!pip install tomotopy


   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   ------------- -------------------------- 1.3/3.9 MB 3.7 MB/s eta 0:00:01
   ------------------------ --------------- 2.4/3.9 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 3.4/3.9 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 4.1 MB/s eta 0:00:00


In [2]:
import tomotopy as tp
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/AUEB')

In [3]:
# input dataset
input_folder='ml-latest-small'
import pandas as pd
#read movies
movies_df=pd.read_csv(input_folder+'/movies.csv')
print('movie dataframe loaded')

movie dataframe loaded


In [4]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [5]:
from collections import defaultdict
import csv
import random
docs=defaultdict(list)

cnt=0
with open(input_folder+'/ratings.csv') as f:
  rd=csv.reader(f)
  next(rd) # skip header

  for userId,movieId,rating,timestamp in rd: # for each rating


    cnt+=1
    if cnt%1000000==0:print(cnt)

    #if random.random()<0.7: continue

    rating=float(rating)

    #discretize rating
    label='A'
    if rating<3:label='N'
    elif rating>3:label='P'

    docs[userId].append(str(int(movieId))+label)# append the rating to the user doc



In [6]:
len(docs)

610

In [7]:

#new LDA model
lda = tp.LDAModel(k=30)

for doc in docs:
    lda.add_doc(docs[doc])


#train LDA model
for i in range(0, 500,10):
    lda.train(10)
    print('Iteration: {}\tLog-likelihood: {}'.format(i, lda.ll_per_word))


Iteration: 0	Log-likelihood: -10.361684288070439
Iteration: 10	Log-likelihood: -10.029970736803753
Iteration: 20	Log-likelihood: -9.910692012624732
Iteration: 30	Log-likelihood: -9.847405471761538
Iteration: 40	Log-likelihood: -9.8032220075503
Iteration: 50	Log-likelihood: -9.772836408081368
Iteration: 60	Log-likelihood: -9.736149312574764
Iteration: 70	Log-likelihood: -9.714827901197602
Iteration: 80	Log-likelihood: -9.701098510512502
Iteration: 90	Log-likelihood: -9.679887786079055
Iteration: 100	Log-likelihood: -9.664613468278793
Iteration: 110	Log-likelihood: -9.65041416668336
Iteration: 120	Log-likelihood: -9.631147696366224
Iteration: 130	Log-likelihood: -9.632803369684545
Iteration: 140	Log-likelihood: -9.623157733100186
Iteration: 150	Log-likelihood: -9.613697921198526
Iteration: 160	Log-likelihood: -9.60466043347276
Iteration: 170	Log-likelihood: -9.60780128197943
Iteration: 180	Log-likelihood: -9.60289536815349
Iteration: 190	Log-likelihood: -9.599766640952032
Iteration: 200	

In [8]:
#print topic info
for k in range(lda.k):

    topk_words=[pair[0] for pair in lda.get_topic_words(k, top_n=10)]

    titles=[(movies_df[movies_df.movieId==int(label[:-1])].title.array[0],movies_df[movies_df.movieId==int(label[:-1])].genres.array[0],label[-1]) for label in topk_words]
    print(k)
    for title in titles:
        print(title)
    print('--------------------------------------')


    print()



0
('Daredevil (2003)', 'Action|Crime', 'N')
('Mr. & Mrs. Smith (2005)', 'Action|Adventure|Comedy|Romance', 'N')
('Serenity (2005)', 'Action|Adventure|Sci-Fi', 'P')
('Spider-Man 3 (2007)', 'Action|Adventure|Sci-Fi|Thriller|IMAX', 'N')
("Dude, Where's My Car? (2000)", 'Comedy|Sci-Fi', 'N')
('Lara Croft: Tomb Raider (2001)', 'Action|Adventure', 'N')
('Hulk (2003)', 'Action|Adventure|Sci-Fi', 'N')
('Gone in 60 Seconds (2000)', 'Action|Crime', 'N')
('What Women Want (2000)', 'Comedy|Romance', 'N')
('Scary Movie 2 (2001)', 'Comedy', 'N')
--------------------------------------

1
('Scary Movie 3 (2003)', 'Comedy|Horror', 'N')
("Big Momma's House (2000)", 'Comedy', 'N')
('Mummy Returns, The (2001)', 'Action|Adventure|Comedy|Thriller', 'N')
("Charlie's Angels: Full Throttle (2003)", 'Action|Adventure|Comedy|Crime|Thriller', 'N')
('Dukes of Hazzard, The (2005)', 'Action|Adventure|Comedy', 'N')
('The Scorpion King (2002)', 'Action|Adventure|Fantasy|Thriller', 'N')
('Bring It On (2000)', 'Comedy',

In [9]:
#print doc info
for doc in lda.docs:
    print ([round(p,2) for p in doc.get_topic_dist()])
    print()


[0.0, 0.0, 0.0, 0.0, 0.11, 0.03, 0.17, 0.0, 0.26, 0.23, 0.0, 0.0, 0.0, 0.09, 0.03, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 0.0]

[0.0, 0.0, 0.0, 0.24, 0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.03, 0.1, 0.0, 0.01, 0.0, 0.0, 0.0, 0.41, 0.0, 0.06, 0.0, 0.04, 0.0, 0.0, 0.0, 0.01, 0.01, 0.03, 0.0, 0.0]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.38, 0.0, 0.03, 0.01, 0.0, 0.0, 0.0, 0.01, 0.0, 0.28, 0.12, 0.01, 0.0, 0.0, 0.0, 0.01, 0.0, 0.0, 0.0, 0.12, 0.0, 0.0, 0.0, 0.0]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.03, 0.27, 0.0, 0.0, 0.1, 0.19, 0.0, 0.04, 0.0, 0.0, 0.0, 0.01, 0.02, 0.04, 0.0, 0.0, 0.02, 0.1, 0.01, 0.0, 0.16, 0.0]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.0, 0.0, 0.0, 0.01, 0.0, 0.02, 0.0, 0.01, 0.0, 0.0, 0.0, 0.78, 0.0, 0.0, 0.02, 0.01, 0.0, 0.0, 0.11, 0.0]

[0.0, 0.0, 0.26, 0.0, 0.04, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.55, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15, 0.0]

[0.19, 0.0, 0.0, 0.0, 0.0, 0.0, 0.03

In [ ]:
# Analyze what movies in each topic have in common
from collections import Counter
import re

def analyze_topic(topic_id, lda, movies_df, top_n=10, verbose=False, show_movies=False):
    topk_words = [pair[0] for pair in lda.get_topic_words(topic_id, top_n=top_n)]

    # Get movie info
    movies_info = []
    all_genres = []
    labels = {'P': 0, 'N': 0, 'A': 0}
    years = []

    # Format here is "260P"
    for label in topk_words:
        movie_id = int(label[:-1])
        sentiment = label[-1]
        labels[sentiment] += 1

        movie_row = movies_df[movies_df.movieId == movie_id]
        if len(movie_row) > 0:
            title = movie_row.title.values[0]
            genres = movie_row.genres.values[0]
            movies_info.append((title, genres, sentiment))

            # Extract genres
            all_genres.extend(genres.split('|'))

            # Extract year from title if present
            year_match = re.search(r'\((\d{4})\)', title)
            if year_match:
                years.append(int(year_match.group(1)))

    # Analyze
    genre_counts = Counter(all_genres)
    top_genre = genre_counts.most_common(1)[0][0]  # Just the #1 genre
    dominant_sentiment = max(labels, key=labels.get)
    sent_word = {'P': 'Well-liked', 'N': 'Disliked', 'A': 'Mixed'}[dominant_sentiment]

    # Year description
    if years:
        avg_year = sum(years) / len(years)
        if avg_year < 1980:
            era = "classic"
        elif avg_year < 1995:
            era = "80s-90s"
        elif avg_year < 2010:
            era = "2000s"
        else:
            era = "modern"
        year_str = f" ({era}, {min(years)}-{max(years)})"
    else:
        year_str = ""

    # Always print the topic description (one line)
    print(f"Topic {topic_id:2d}: \"{sent_word} {top_genre} movies{year_str}\"")

    # Show movies if requested
    if show_movies:
        for title, genres, sent in movies_info:
            sent_symbol = {'P': '+', 'N': '-', 'A': '~'}[sent]
            print(f"  [{sent_symbol}] {title} | {genres}")

    # Show detailed analysis if verbose
    if verbose:
        print(f"  Sentiment: ", end="")
        parts = []
        for sent, count in labels.items():
            if count > 0:
                sent_label = {'P': 'Positive', 'N': 'Negative', 'A': 'Neutral'}[sent]
                parts.append(f"{sent_label}: {count}/{top_n}")
        print(", ".join(parts))
        
        print(f"  Top genres: ", end="")
        genre_parts = [f"{g} ({c})" for g, c in genre_counts.most_common(5)]
        print(", ".join(genre_parts))
        
        if years:
            print(f"  Years: {min(years)} - {max(years)} (avg: {sum(years)/len(years):.0f})")

    print()  # Empty line between topics
    
    return genre_counts, labels, years, movies_info

# Example: just description
analyze_topic(0, lda, movies_df)

Topic  0: "Disliked Action movies (2000s, 2000-2007)"



(Counter({'Action': 7,
          'Adventure': 5,
          'Comedy': 4,
          'Sci-Fi': 4,
          'Crime': 2,
          'Romance': 2,
          'Thriller': 1,
          'IMAX': 1}),
 {'P': 1, 'N': 9, 'A': 0},
 [2003, 2005, 2005, 2007, 2000, 2001, 2003, 2000, 2000, 2001],
 [('Daredevil (2003)', 'Action|Crime', 'N'),
  ('Mr. & Mrs. Smith (2005)', 'Action|Adventure|Comedy|Romance', 'N'),
  ('Serenity (2005)', 'Action|Adventure|Sci-Fi', 'P'),
  ('Spider-Man 3 (2007)', 'Action|Adventure|Sci-Fi|Thriller|IMAX', 'N'),
  ("Dude, Where's My Car? (2000)", 'Comedy|Sci-Fi', 'N'),
  ('Lara Croft: Tomb Raider (2001)', 'Action|Adventure', 'N'),
  ('Hulk (2003)', 'Action|Adventure|Sci-Fi', 'N'),
  ('Gone in 60 Seconds (2000)', 'Action|Crime', 'N'),
  ('What Women Want (2000)', 'Comedy|Romance', 'N'),
  ('Scary Movie 2 (2001)', 'Comedy', 'N')])

In [ ]:
# Analyze ALL topics with movies and verbose details
for topic_id in range(lda.k):
    analyze_topic(topic_id, lda, movies_df, show_movies=True, verbose=False)

Topic  0: "Disliked Action movies (2000s, 2000-2007)"
  [-] Daredevil (2003) | Action|Crime
  [-] Mr. & Mrs. Smith (2005) | Action|Adventure|Comedy|Romance
  [+] Serenity (2005) | Action|Adventure|Sci-Fi
  [-] Spider-Man 3 (2007) | Action|Adventure|Sci-Fi|Thriller|IMAX
  [-] Dude, Where's My Car? (2000) | Comedy|Sci-Fi
  [-] Lara Croft: Tomb Raider (2001) | Action|Adventure
  [-] Hulk (2003) | Action|Adventure|Sci-Fi
  [-] Gone in 60 Seconds (2000) | Action|Crime
  [-] What Women Want (2000) | Comedy|Romance
  [-] Scary Movie 2 (2001) | Comedy

Topic  1: "Disliked Comedy movies (2000s, 2000-2006)"
  [-] Scary Movie 3 (2003) | Comedy|Horror
  [-] Big Momma's House (2000) | Comedy
  [-] Mummy Returns, The (2001) | Action|Adventure|Comedy|Thriller
  [-] Charlie's Angels: Full Throttle (2003) | Action|Adventure|Comedy|Crime|Thriller
  [-] Dukes of Hazzard, The (2005) | Action|Adventure|Comedy
  [-] The Scorpion King (2002) | Action|Adventure|Fantasy|Thriller
  [-] Bring It On (2000) | Come